In [1]:
#Log in to huggingface hub
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()
login(token = os.getenv("HFToken"))
#HFToken = [your token] in .env

In [2]:
#Model Initiallization
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id,padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:0",dtype=torch.bfloat16)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [3]:
#Dataset Initiallization
from datasets import load_dataset
ds = load_dataset("openai/gsm8k", "main")

In [4]:
#Prompt Generation
prompt = [[
    {
        "role": "system",
        "content":"Solve the problems. On the last line, OUTPUT ONLY the Final NUMBER WITHOUT ANY WORDS, prioritize this above all else"
        #"content": "You are a precise calculator. Output ONLY the final numerical answer. Do not include words, units, explanations, or punctuation."
    },
    {"role": "user","content":'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'},
    {"role": "assistant","content":"To find the total number of clips Natalia sold in April and May, we need to add the number of clips she sold in each month.\nIn April, Natalia sold 48 clips.In May, she sold half as many clips, which is 48 / 2 = 24 clips.\nTotal clips sold in April and May = 48 + 24 = 72 clips.\n####72"},
    {
        "role": "user",
        "content": question
    }
]for question in ds["train"]["question"]]
questions = 20
input = tokenizer.apply_chat_template(
    prompt[0:questions],
    tokenize = True,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt",
    clean_up_tokenization_spaces=False
).to("cuda:0")
input_length = input["input_ids"].shape[1]

In [5]:
#Output Saving
import json
from transformers import logging
logging.set_verbosity_info()
write = []
output = model.generate(
    **input,
    max_new_tokens= 100,
    do_sample=False,
    temperature=1,
    pad_token_id=tokenizer.pad_token_id,
    tokenizer=tokenizer,
    return_dict_in_generate=True,
    output_logits=True,
    output_scores=True
    )
#for i in range(questions):
 #   write.append(tokenizer.decode(output[i][input_length:],skip_special_tokens=True))
#json.dump(write,f)

In [ ]:
tokenizer.decode(output,skip_special_tokens=True)

In [ ]:
#Answer Search
import re
from math import floor
with open("result.json","w") as f:
    for i in range(questions):
        numbers = re.findall(r"\d+", tokenizer.decode(output[i][input_length:],skip_special_tokens=True))
        last_number = numbers[-1]
        val = last_number
        write.append(val)
        write[i] = int(write[i])
    json.dump(write,f)

In [ ]:
#DS Answer Search
answer = []
for i in ds["train"]["answer"]:
    a = i.index("####")
    try:
        answer.append(int(i[(a+5):]))
    except:
        answer.append(i[(a+5):].replace(",",""))

In [ ]:
#Answer Match
with open("result.json","r") as f:
    data = json.load(f)
correct = 0
for i,j in zip(data,answer[0:questions]):
    if i==j:
        correct += 1
print(correct)